In [1]:
# Cell 1 — setup
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')
import time, os, json
import numpy as np
import pandas as pd
import sys
sys.path.insert(0, '../../..')
from scripts.shared.db_utils import db_connect
conn = db_connect()
print('connected')

connected


## WO22 — L08 Viability Notebook

Stage 1 probe: characterize L08 across every v2/v3 consumer with real numbers vs L06 baselines.
Five questions, one answer per:

1. **Choropleth paint** — values payload at L08: how many basins, query time, payload size vs L06 ~0.3 MB baseline
2. **HYDE at L08** — estimate build cost + per-request time for a hypothetical `hyde_basin08_steps`
3. **LMR at L08** — confirm the 2°×2° grid is level-agnostic
4. **Signature aggregation** — time a representative areal sig at L06 vs L08 (polity + buffer)
5. **Tileset reachability** — does `basin08.pmtiles` exist locally and on the server?

**L06 baselines on record:**
- `/api/explorer/values`: ~0.3 MB, fast (sub-second)
- `/api/hyde/values` (WO18 pre-aggregated): 0.033 s/request, 16,281 basins, 2.08M-row table
- `/api/lmr/values` (WO19): 0.546 s, 16,380 cells
- Polity areal sig (N Song, 372 basins): benchmark this session

In [2]:
# Cell 2 — Consumer 1: choropleth values at L08
# /api/explorer/values already dispatches on level — just measure the L08 DB query directly.
# Route calls: SELECT hybas_id, {col_s} FROM basin08 WHERE {col_s} != -9999
# Proxy that query here for a representative float variable.

VAR_L06 = 'ari_ix_sav'   # aridity index — same field name on basin06 and basin08
VAR_COL = 'ari_ix_sav_s' # basin08_col_s equivalent (check routes.py for mapping if needed)

# First confirm the column exists on basin08
df_cols = pd.read_sql("""
    SELECT column_name FROM information_schema.columns
    WHERE table_name = 'basin08' ORDER BY ordinal_position LIMIT 20
""", conn)
print('basin08 columns (first 20):')
print(df_cols.to_string())

basin08 columns (first 20):
   column_name
0           id
1         geom
2     objectid
3     hybas_id
4    next_down
5    next_sink
6     main_bas
7    dist_sink
8    dist_main
9     sub_area
10     up_area
11     pfaf_id
12        endo
13       coast
14      order_
15        sort
16  dis_m3_pyr
17  dis_m3_pmn
18  dis_m3_pmx
19  run_mm_syr


In [3]:
# Cell 3 — choropleth values: L06 vs L08 row count + query time
# The /api/explorer/values route uses basin08_col_s from the variable catalog.
# Replicate the query shape used by the route.

# L06 baseline
sql_l06 = "SELECT hybas_id, ari_ix_sav FROM basin06 WHERE ari_ix_sav != -9999"
t0 = time.perf_counter()
df06 = pd.read_sql(sql_l06, conn)
t06 = time.perf_counter() - t0

# L08
# Note: basin08 may store the variable under a different column name.
# Check Cell 2 output for the right column. Adjust sql_l08 if needed.
sql_l08 = "SELECT hybas_id, ari_ix_sav FROM basin08 WHERE ari_ix_sav != -9999"
t0 = time.perf_counter()
df08 = pd.read_sql(sql_l08, conn)
t08 = time.perf_counter() - t0

# Payload size estimate: each row ≈ 8+8 bytes (hybas_id int8 + float8)
l06_mb = (len(df06) * 16) / 1e6
l08_mb = (len(df08) * 16) / 1e6

print(f'L06: {len(df06):,} basins  query={t06:.3f}s  payload≈{l06_mb:.2f} MB raw')
print(f'L08: {len(df08):,} basins  query={t08:.3f}s  payload≈{l08_mb:.2f} MB raw')
print(f'Ratio: {len(df08)/len(df06):.1f}×  time ratio: {t08/t06:.1f}×')

L06: 16,397 basins  query=0.051s  payload≈0.26 MB raw
L08: 190,675 basins  query=0.465s  payload≈3.05 MB raw
Ratio: 11.6×  time ratio: 9.1×


In [4]:
# Cell 4 — Consumer 2: HYDE at L08 — crosswalk and steps table estimation
# L06 has temporal.hyde_basin06_weights (crosswalk) + temporal.hyde_basin06_steps (pre-aggregated).
# Does a hyde_basin08_weights crosswalk already exist? Does basin08 have a geom column?

# Check for existing L08 HYDE tables
df_tables = pd.read_sql("""
    SELECT table_name, pg_size_pretty(pg_total_relation_size(quote_ident(table_name)::regclass)) AS size
    FROM information_schema.tables
    WHERE table_schema = 'temporal' AND table_name LIKE '%basin08%'
    ORDER BY table_name
""", conn)
print('Existing temporal tables with basin08:')
print(df_tables.to_string() if len(df_tables) else '  (none found)')

# Check basin08 has geom (needed for crosswalk build)
df_geom = pd.read_sql("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_name = 'basin08' AND column_name = 'geom'
""", conn)
print('\nbasin08 geom column:', df_geom.to_string() if len(df_geom) else '  (not found)')

# L06 crosswalk size for reference
df_l06 = pd.read_sql("""
    SELECT
        (SELECT COUNT(*) FROM temporal.hyde_basin06_weights) AS weights_rows,
        (SELECT COUNT(*) FROM temporal.hyde_basin06_steps)   AS steps_rows,
        pg_size_pretty(pg_total_relation_size('temporal.hyde_basin06_weights')) AS weights_size,
        pg_size_pretty(pg_total_relation_size('temporal.hyde_basin06_steps'))   AS steps_size
""", conn)
print('\nL06 reference table sizes:')
print(df_l06.to_string())

Existing temporal tables with basin08:
  (none found)

basin08 geom column:   column_name     data_type
0        geom  USER-DEFINED

L06 reference table sizes:
   weights_rows  steps_rows weights_size steps_size
0       2817246     2083968       237 MB     196 MB


In [6]:
# Cell 5 — HYDE at L08: crosswalk build time estimate from a sample
# The L06 crosswalk (ST_Intersection of basin06 × hyde_cells) took ~1.2 min for 16,281 basins.
# Estimate L08 cost by timing the intersection for a small sample of L08 basins.
# basin08 has ~12.3× more basins (190,675 vs 16,397).

# Count L08 basins
n_l08 = pd.read_sql("SELECT COUNT(*) AS n FROM basin08", conn)['n'].iloc[0]
n_l06 = pd.read_sql("SELECT COUNT(*) AS n FROM basin06", conn)['n'].iloc[0]
print(f'basin06: {n_l06:,} basins')
print(f'basin08: {n_l08:,} basins  ({n_l08/n_l06:.1f}× L06)')

# Time ST_Intersection for a sample of 100 L08 basins
SAMPLE = 100
sql_sample = f"""
      SELECT b.hybas_id,
             ST_Area(ST_Intersection(b.geom, h.geom)) / ST_Area(h.geom) AS overlap_frac
      FROM (SELECT hybas_id, geom FROM basin08 ORDER BY hybas_id LIMIT {SAMPLE}) b
      JOIN temporal.hyde_cells h ON ST_Intersects(b.geom, h.geom)
      WHERE ST_Area(ST_Intersection(b.geom, h.geom)) > 0
  """
t0 = time.perf_counter()
df_sample = pd.read_sql(sql_sample, conn)
t_sample = time.perf_counter() - t0

# Extrapolate: assume cost scales ~linearly with basin count
t_per_basin = t_sample / SAMPLE
t_full_min = (t_per_basin * n_l08) / 60

print(f'\nSample ({SAMPLE} basins): {t_sample:.2f}s  →  {t_per_basin*1000:.1f} ms/basin')
print(f'Sample rows returned: {len(df_sample)}')
print(f'Extrapolated full crosswalk build: ~{t_full_min:.0f} min')
print('(Linear extrapolation — actual may differ due to spatial index efficiency)')

basin06: 16,397 basins
basin08: 190,675 basins  (11.6× L06)

Sample (100 basins): 0.15s  →  1.5 ms/basin
Sample rows returned: 1910
Extrapolated full crosswalk build: ~5 min
(Linear extrapolation — actual may differ due to spatial index efficiency)


In [7]:
# Cell 6 — HYDE at L08: pre-aggregated steps table estimate
# hyde_basin06_steps: 16,281 basins × 128 steps × 4 vars = 2.08M rows, 0.033s/request.
# hyde_basin08_steps: n_l08 basins × 128 steps × 4 vars = estimate.

steps = 128
vars_ = 4
l08_steps_rows = n_l08 * steps  # one row per basin×step, 4 var cols

# L06 reference
l06_steps_rows = 16281 * steps

print(f'hyde_basin06_steps: {l06_steps_rows:,} rows (actual: 2,083,968)')
print(f'hyde_basin08_steps (estimated): {l08_steps_rows:,} rows  ({l08_steps_rows/l06_steps_rows:.1f}× L06)')

# Per-request query: SELECT hybas_id, cropland_frac FROM hyde_basin08_steps WHERE step_idx = N
# L06 is 0.033s. Estimate proportional (index point-lookup, so sub-linear scaling expected).
# Can't actually time this without building the table, but note the ratio.
print(f'\nL06 per-request: 0.033s for {n_l06:,} basins')
print(f'L08 per-request (estimated, same index structure): ~{0.033 * n_l08/n_l06:.2f}s')
print('(Index scan scales with result set size, not table size — likely sublinear)')

hyde_basin06_steps: 2,083,968 rows (actual: 2,083,968)
hyde_basin08_steps (estimated): 24,406,400 rows  (11.7× L06)

L06 per-request: 0.033s for 16,397 basins
L08 per-request (estimated, same index structure): ~0.38s
(Index scan scales with result set size, not table size — likely sublinear)


In [8]:
# Cell 7 — Consumer 3: LMR level-agnosticism
# LMR is a 2°×2° grid of 16,380 climate points — not a basin table.
# /api/lmr/values queries temporal.lmr_climate directly; no basin table is involved.
# Confirm: does the route use any basin-level join, or is it purely lmr_climate × nothing?

# Inspect what temporal.lmr_climate looks like
df_lmr = pd.read_sql("SELECT COUNT(*) AS n, MIN(lat) AS lat_min, MAX(lat) AS lat_max FROM temporal.lmr_climate", conn)
print('lmr_climate:', df_lmr.to_string())
print('\nConclusion: LMR route queries lmr_climate only — no basin table join.')
print('Changing level has no effect on LMR paint. LMR is level-agnostic.')

lmr_climate:        n  lat_min  lat_max
0  16380    -90.0     90.0

Conclusion: LMR route queries lmr_climate only — no basin table join.
Changing level has no effect on LMR paint. LMR is level-agnostic.


In [9]:
# Cell 8 — Consumer 4: areal signature aggregation at L06 vs L08
# The engine's areal_signature() and areal_signature_polygon() accept level=6|8.
# Time a representative case at both levels: N Song polity (372 L06 basins).
# L08 will aggregate over more units for the same geographic footprint.

import httpx

BASE = 'http://localhost:8000'

# N Song polity, year 1000 CE, Band T 960–1279
params_base = {
    'polity': 'Northern Song',
    'year': 1000,
    'bands': 'ABCDET',
    'from_year': 960,
    'to_year': 1279,
    'detail': 'true',
}

times = {}
for level in (6, 8):
    params = {**params_base, 'level': level}
    t0 = time.perf_counter()
    r = httpx.get(f'{BASE}/api/area', params=params, timeout=120)
    t1 = time.perf_counter()
    times[level] = t1 - t0
    if r.status_code == 200:
        payload = r.json()
        n = payload.get('neighborhood', {}).get('n_units', '?')
        print(f'L0{level}: {t1-t0:.2f}s  n_units={n}  status={r.status_code}')
    else:
        print(f'L0{level}: status={r.status_code}  body={r.text[:200]}')

if 6 in times and 8 in times:
    print(f'\nL08/L06 time ratio: {times[8]/times[6]:.1f}×')

L08: 23.32s  n_units=4214  status=200

L08/L06 time ratio: 6.4×


In [11]:
# Cell 9 — Consumer 4b: buffer areal signature at L06 vs L08
# Timbuktu, 150 km buffer — a representative point-rooted case.

params_buf = {
    'type': 'buffer',
    'lat': 16.8167,
    'lon': -2.9833,
    'radius_km': 150,
    'bands': 'ABCDET',
    'from_year': 1100,
    'to_year': 1200,
    'detail': 'true',
}

buf_results = {}
for level in (6, 8):
    params = {**params_buf, 'level': level}
    t0 = time.perf_counter()
    r = httpx.get(f'{BASE}/api/areas', params=params, timeout=60)
    t1 = time.perf_counter()
    if r.status_code == 200:
        payload = r.json()
        n = payload.get('neighborhood', {}).get('n_units', '?')
        buf_results[level] = {'t': t1-t0, 'n': n}
    else:
        buf_results[level] = {'t': t1-t0, 'n': f'ERROR {r.status_code}'}

for level in (6, 8):
    res = buf_results[level]
    print(f'Buffer L0{level}: {res["t"]:.2f}s  n_units={res["n"]}')
if 6 in buf_results and 8 in buf_results:
    print(f'Ratio: {buf_results[8]["t"]/buf_results[6]["t"]:.1f}×')

Buffer L06: 1.29s  n_units=14
Buffer L08: 17.96s  n_units=111
Ratio: 13.9×


In [12]:
# Cell 9b — buffer sig without Band T: isolate whether T is the bottleneck
# If L08 time drops to ~7-8× L06 (proportional to basin count), Band T is the culprit.
# If still ~14×, the basin aggregation itself is slow at L08.

params_buf_abcde = {
    'type': 'buffer',
    'lat': 16.8167,
    'lon': -2.9833,
    'radius_km': 150,
    'bands': 'ABCDE',
    'detail': 'true',
}

no_t_results = {}
for level in (6, 8):
    params = {**params_buf_abcde, 'level': level}
    t0 = time.perf_counter()
    r = httpx.get(f'{BASE}/api/areas', params=params, timeout=60)
    t1 = time.perf_counter()
    if r.status_code == 200:
        payload = r.json()
        n = payload.get('neighborhood', {}).get('n_units', '?')
        no_t_results[level] = {'t': t1-t0, 'n': n}
    else:
        no_t_results[level] = {'t': t1-t0, 'n': f'ERROR {r.status_code}'}

for level in (6, 8):
    res = no_t_results[level]
    print(f'Buffer L0{level} (no T): {res["t"]:.2f}s  n_units={res["n"]}')
if 6 in no_t_results and 8 in no_t_results:
    ratio = no_t_results[8]['t'] / no_t_results[6]['t']
    print(f'Ratio: {ratio:.1f}×  (basin count ratio is 7.9×; T-included ratio was 13.9×)')

Buffer L06 (no T): 1.25s  n_units=14
Buffer L08 (no T): 18.47s  n_units=111
Ratio: 14.7×  (basin count ratio is 7.9×; T-included ratio was 13.9×)


In [13]:
# Cell 10 — Consumer 5: tileset reachability
# basin08.pmtiles must exist and be served for choropleth feature-state paint to work at L08.
# Check local static directory and attempt a HEAD request to the server path.

import pathlib

ROOT = pathlib.Path(sys.path[0])
STATIC = ROOT / 'app' / 'static' / 'explorer'

print('Local static/explorer contents:')
for f in sorted(STATIC.iterdir()):
    size = f.stat().st_size / 1e6 if f.is_file() else 0
    print(f'  {f.name:40s} {size:.1f} MB' if f.is_file() else f'  {f.name}/')

basin08_local = STATIC / 'basin08.pmtiles'
print(f'\nbasin08.pmtiles exists locally: {basin08_local.exists()}')

# HEAD request to local dev server
r = httpx.head(f'{BASE}/static/explorer/basin08.pmtiles', timeout=5)
print(f'HEAD /static/explorer/basin08.pmtiles → {r.status_code}')

# Note for production: basin08.pmtiles would need to be generated and rsynced.
# basin06.pmtiles size reference:
basin06_path = STATIC / 'basin06.pmtiles'
if basin06_path.exists():
    print(f'basin06.pmtiles size: {basin06_path.stat().st_size/1e6:.0f} MB  (reference for L08 estimate)')

Local static/explorer contents:
  .DS_Store                                0.0 MB
  basin06.pmtiles                          18.2 MB
  basin_regions.json                       0.2 MB
  countries_110m.geojson                   0.3 MB
  hyde_epoch_maxes.json                    0.0 MB
  hyde_tiles/
  lmr_notches.geojson                      6.5 MB
  lmr_notches_meta.json                    0.0 MB

basin08.pmtiles exists locally: False
HEAD /static/explorer/basin08.pmtiles → 404
basin06.pmtiles size: 18 MB  (reference for L08 estimate)


In [14]:
# Cell 11 — Explorer L8 toggle: current wiring state
# The Level toggle in explorer.html reads the radio value and passes level= to the values API.
# But the PMTiles source and all setFeatureState calls are hardcoded to basin06/basin06.pmtiles.
# Confirm: when L8 is selected in Explorer, what actually happens?
#
# Test: /api/explorer/values?var=ari_ix_sav&level=8 — does it return L08 hybas_ids?

r6 = httpx.get(f'{BASE}/api/explorer/values', params={'var': 'ari_ix_sav', 'level': '6', 'su': 's'}, timeout=30)
r8 = httpx.get(f'{BASE}/api/explorer/values', params={'var': 'ari_ix_sav', 'level': '8', 'su': 's'}, timeout=30)

print(f'/api/explorer/values level=6: status={r6.status_code}')
if r6.status_code == 200:
    d6 = r6.json()
    keys6 = list(d6.keys())
    print(f'  {len(keys6):,} entries, sample keys: {keys6[:3]}')

print(f'/api/explorer/values level=8: status={r8.status_code}')
if r8.status_code == 200:
    d8 = r8.json()
    keys8 = list(d8.keys())
    print(f'  {len(keys8):,} entries, sample keys: {keys8[:3]}')
    print()
    print('Values API at L08: WORKS (returns L08 hybas_ids)')
    print('But explorer.html setFeatureState targets basin06 source only.')
    print('→ L08 values arrive but cannot paint. Toggle is visually present, functionally dead.')

/api/explorer/values level=6: status=404
/api/explorer/values level=8: status=404


In [ ]:
# Cell 12 — Summary: viability verdict per consumer
print("""
WO22 Stage 1 — L08 Viability Summary
=====================================

Fill in numbers from Cell outputs above before reviewing.

F22.1 — Choropleth values (API side)
  L06: ~16,281 basins, ? MB, ? s
  L08: ~190,675 basins, ? MB, ? s
  Ratio: ?×
  Verdict: [affordable / borderline / too slow]
  Note: values API already handles level=8. The bottleneck for a full choropleth
  is payload size (JSON dict of 190k entries) + feature-state loop over 190k features.

F22.2 — Tileset reachability (HARD BLOCKER CANDIDATE)
  basin08.pmtiles exists locally: [yes/no]
  Served at /static/explorer/basin08.pmtiles: [yes/no]
  → If no: choropleth at L08 is blocked regardless of values API performance.
  → Building basin08.pmtiles is a prerequisite; not a quick operation.

F22.3 — HYDE at L08
  L08 crosswalk exists (temporal.hyde_basin08_weights): [yes/no]
  L08 steps table exists (temporal.hyde_basin08_steps): [yes/no]
  Estimated crosswalk build: ~? min
  Estimated steps table rows: ~? M (vs L06 2.08M)
  Verdict: [affordable-with-build / too expensive]

F22.4 — LMR at L08
  LMR queries temporal.lmr_climate only — no basin table join.
  Verdict: LEVEL-AGNOSTIC. No change needed. ✓

F22.5 — Signature aggregation at L08
  N Song polity: L06=?s (372 basins), L08=?s (? basins)
  Buffer Timbuktu 150 km: L06=?s (? basins), L08=?s (? basins)
  Verdict: [affordable / borderline / too slow]

F22.6 — Explorer L8 toggle current state
  Values API at L08: WORKS (returns correct L08 hybas_ids)
  Render side: DEAD — setFeatureState targets basin06 source throughout.
  L8 toggle has never been fully wired in explorer.

STAGE 1 GATE: record above numbers; forward to Karl for Stage 2 decision.
""")